# S3 J5 - Partage d'état entre agents

Objectifs

Comprendre :

- Agent State
- Shared State
- Session State
- Redis
- PostgreSQL
- Mémoire court terme
- Mémoire long terme
- Coordination multi-agent
```
- Comment plusieurs agents partagent un état ?
- Comment éviter les incohérences ?
- Comment gérer la mémoire ?
- Comment éviter que chaque agent refasse le travail ?
```

## Partie 1 — Pourquoi partager des informations ?

Présentation du problème.

Un agent unique ne pose aucun problème.

Dès que plusieurs agents apparaissent :
```
Planner
Researcher
Writer
Reviewer
```
une question apparaît :

Comment échangent-ils leurs informations ?

## Partie 2 — Workflow et output_key (ADK)

Architecture séquentielle
```
User
↓
Planner
↓
Researcher
↓
Writer
```

Présentation de output_key

In [ ]:
Planner(
    output_key="plan"
)

Researcher(
    input_key="plan",
    output_key="research"
)

Simulation Python

In [ ]:
workflow = {}

workflow["plan"] = planner.run()

workflow["research"] = researcher.run(
    workflow["plan"]
)

Diagramme
```
Planner
↓
output_key="plan"
↓
Researcher
↓
output_key="research"
↓
Writer
```

## Partie 3 — Architecture parallèle (ADK)

Diagramme
```
             Planner

          /      |      \

    Research Weather Finance

          \      |      /

          Aggregator
```

Explication

Chaque agent produit un output_key.

In [ ]:
research

weather

finance

Simulation

In [ ]:
state = {}

state["research"] = ...

state["weather"] = ...

state["finance"] = ...

Aggregator

In [ ]:
report = aggregate(
    state["research"],
    state["weather"],
    state["finance"]
)

## Partie 4 — Limites de output_key

Pourquoi cela devient compliqué ?

Exemple :

Le BudgetAgent modifie
```
budget
```
pendant que

HotelAgent

modifie
```
hotel
```
et

FlightAgent

modifie
```
flight
```

Les agents commencent à partager un véritable état.

## Partie 5 — Shared State

Définition

In [ ]:
trip = {

    "destination":"Rome",

    "budget":1500,

    "hotel":None,

    "flight":None
}

Tous les agents lisent :

```
trip
```
Tous les agents modifient :
```
trip
```

output_key n'est pas réservé aux pipelines séquentiels. Il est également utilisé dans des architectures parallèles, notamment avec un Aggregator (ou un agent de synthèse).

Cependant, il y a toujours une différence conceptuelle entre faire circuler des résultats et maintenir un état partagé.

### Architecture parallèle dans ADK

Tu as probablement vu un schéma de ce type :
```
                  User
                    │
                    ▼
              Coordinator
                    │
      ┌─────────────┼─────────────┐
      ▼             ▼             ▼
 Research Agent  Finance Agent  Weather Agent
      │             │             │
 output_key      output_key    output_key
      │             │             │
      └─────────────┼─────────────┘
                    ▼
               Aggregator
                    │
                    ▼
              Réponse finale
```
Chaque agent travaille indépendamment.

Par exemple :
```
research -> output_key="research"

finance -> output_key="finance"

weather -> output_key="weather"
```
Puis l'Aggregator reçoit quelque chose comme :
```
state = {
    "research": "...",
    "finance": "...",
    "weather": "..."
}
```
Il peut ensuite produire la réponse finale.

On se rapproche du Shared State

Dans ce cas, output_key ressemble déjà à un état partagé.

Mais la philosophie reste différente.

L'état est principalement constitué des résultats produits par les agents.

Par exemple :
```
{
    "research": "...",
    "weather": "...",
    "finance": "..."
}
```
Chaque agent écrit dans sa propre "case".

L'Aggregator lit ensuite ces cases.

Ce qu'un Shared State ajoute

Dans une architecture plus générale, les agents peuvent également partager des informations métier qui évoluent pendant l'exécution.

Par exemple :
```
state = {
    "destination": "Rome",
    "budget": 1500,
    "flight": None,
    "hotel": None,
    "weather": None,
    "constraints": [
        "départ avant 9h"
    ]
}
```
Le WeatherAgent peut enrichir :
```
state["weather"] = "25°C"
```
Le BudgetAgent peut modifier :
```
state["budget"] = 1350
```
Le BookingAgent peut ajouter :
```
state["flight"] = {...}
```
Tous manipulent le même objet logique.

Une autre différence importante : la durée de vie

Dans ADK, les output_key vivent généralement pendant l'exécution du workflow.

Une fois le workflow terminé :
```
output_key
──────────────► supprimé
```
Dans une architecture avec Redis ou PostgreSQL :
```
Workflow terminé
        │
        ▼
Redis
        │
        ▼
PostgreSQL
```
L'état peut être conservé plusieurs heures, plusieurs jours, voire plusieurs mois.

Cela permet de reprendre une tâche interrompue, de partager des informations entre plusieurs workflows ou de conserver des préférences utilisateur.

Le rôle de l'Aggregator

L'Aggregator joue essentiellement le rôle d'un consommateur de résultats.

Il ne décide pas forcément où les données sont stockées.

Il agrège les sorties :
```
final_answer = aggregate(
    research,
    finance,
    weather
)
```
Dans une architecture distribuée, un superviseur peut en plus :

- gérer un état global,
- relancer un agent,
- planifier une nouvelle étape,
- mettre à jour Redis,
- écrire une mémoire durable dans PostgreSQL.

L'Aggregator devient alors un composant parmi d'autres.

# Sharing State Between Agents

## Objectif :

Comprendre comment plusieurs agents
partagent des informations.


# Pourquoi ce sujet est important ?

Agent unique :
```
Utilisateur
↓
Agent
↓
Réponse

Simple.

Multi-agent :

Utilisateur
↓
Supervisor
↓
Planner
↓
Researcher
↓
Writer
```
Qui garde l'état ?

# Définition

State = ensemble des informations nécessaires
à la poursuite d'une tâche.

# Exemple

Utilisateur :

"Prépare un voyage à Rome."

Le Planner produit :

- Vol
- Hôtel
- Activités

Le Researcher doit connaître ce plan.



In [ ]:
task = {

    "destination": "Rome",

    "steps": [
        "flight",
        "hotel",
        "activities"
    ]
}

# Solution naïve

Chaque agent possède sa copie.

Mauvaise idée.

In [ ]:
planner_state = task.copy()
researcher_state = task.copy()
writer_state = task.copy()

# Pourquoi c'est mauvais ?

Les copies divergent rapidement.

In [ ]:
planner_state["budget"] = 1200

print(researcher_state)

# Shared State

Tous les agents lisent
une source unique.

In [ ]:
shared_state = {

    "destination": "Rome",

    "budget": None,

    "hotel": None
}

class Planner:

    def run(self):

        shared_state["budget"] = 1200

class Researcher:

    def run(self):

        return shared_state["budget"]

Planner().run()

Researcher().run()



# Architecture classique
```
           Redis

             ▲

   ┌─────────┼─────────┐

   ▼         ▼         ▼

Planner  Researcher  Writer
```

# Pourquoi Redis ?

- très rapide
- partagé
- simple
- TTL

In [ ]:
import redis

redis_client = redis.Redis(
    host="localhost",
    port=6379
)

redis_client.set(
    "session:123:budget",
    "1200"
)

redis_client.get(
    "session:123:budget"
)

# Structure recommandée

In [ ]:
session_state = {

    "user_id": "123",

    "conversation_id": "abc",

    "current_task": "trip",

    "budget": 1200
}

# Session State

Durée :

minutes ou heures.

# Long Term Memory

Durée :

jours
mois
années

# Exemple

Préférence utilisateur :

"Je préfère Air France."

Doit être sauvegardée.

```
CREATE TABLE memories (

    id BIGSERIAL PRIMARY KEY,

    user_id TEXT,

    content TEXT,

    created_at TIMESTAMP
);
```

# Redis vs PostgreSQL

Redis

→ état courant

PostgreSQL

→ mémoire durable

# Pattern Supervisor

In [ ]:
class Supervisor:

    def __init__(self):

        self.state = shared_state

    def run(self):

        Planner().run()

        Researcher().run()

# Que faut-il stocker ?

In [ ]:
important_keywords = [

    "objectif",

    "préférence",

    "projet",

    "contrainte"
]

def should_store(text):

    return any(
        word in text.lower()
        for word in important_keywords
    )

# Conflits d'écriture

Deux agents peuvent modifier
la même donnée.

In [ ]:
shared_state["budget"] = 1200

shared_state["budget"] = 1500

# Solutions

- verrou Redis
- versioning
- event sourcing

# Architecture Production

```
Utilisateur
      │
      ▼
Supervisor
      │
 ┌────┼─────┬─────┐
 ▼    ▼     ▼     ▼
Plan Res  Write Review
      │
      ▼
Redis Shared State
      │
      ▼
PostgreSQL Memory
```

# Observabilité

In [ ]:
import time

start = time.time()

Planner().run()

duration = time.time() - start

print(duration)

# Anti-patterns

❌ Chaque agent possède sa mémoire

❌ Sauvegarder toute la conversation

❌ Contexte illimité

❌ Pas de TTL

# Mini Projet

Construire :

- Planner
- Researcher
- Writer

partageant Redis.

# Exercice 1

Créer un shared_state Redis.

# Exercice 2

Ajouter un Supervisor.

# Exercice 3

Sauvegarder les préférences utilisateur.

# Exercice 4

Créer une mémoire PostgreSQL.

# Exercice 5

Dessiner l'architecture complète :
```
OpenAI
↓
Supervisor
↓
Agents
↓
Redis
↓
PostgreSQL
```

# Notes d'architecte

La plupart des développeurs pensent :

"Le problème est le prompt."

En production, le problème est souvent :

- l'état
- la mémoire
- la cohérence
- la coordination

C'est là que se situe une grande partie de la valeur d'un AI Engineer.